In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import SVR
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout


In [13]:
# Load your dataset (adjust the path to where your file is located)
data = pd.read_csv('../../data/Area1.csv')

In [14]:
data

,Date,AVG_NO_USER,AVG_USR_THRPUT_DL,DL_TRAFFIC_MB
0,2022-09-01 00:00:00,6919.59,649.73,15926000.77
1,2022-09-02 00:00:00,6916.80,716.17,15888133.90
2,2022-09-03 00:00:00,6852.06,740.46,14982246.10
3,2022-09-04 00:00:00,6846.14,715.02,15768670.80
4,2022-09-05 00:00:00,6784.53,781.21,15351915.55
...,...,...,...,...
361,2023-08-28 00:00:00,9044.90,3267.01,25373601.86
362,2023-08-29 00:00:00,8892.82,3037.00,23986359.10
363,2023-08-30 00:00:00,8904.82,3130.67,24443806.05
364,2023-08-31 00:00:00,8978.82,3127.35,25278187.70


In [15]:
data.columns

Index(['Date', 'AVG_NO_USER', 'AVG_USR_THRPUT_DL', 'DL_TRAFFIC_MB'], dtype='object')

In [16]:
# List of the first 10 features based on dataset
selected_features = [
    'AVG_USR_THRPUT_DL', 'DL_TRAFFIC_MB'
]

# Target feature
target_feature = 'AVG_NO_USER'

# Normalize all features
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data[[target_feature] + selected_features])

# Split data into train and test sets (80% train, 20% test)
train_size = int(len(data_scaled) * 0.8)
train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]

# Function to create dataset matrix for GRU
def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), :])
        Y.append(dataset[i + look_back, 0])  # Target is the first column after scaling
    return np.array(X), np.array(Y)

# Function to train GRU model
def train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer):
    model = Sequential()
    model.add(GRU(units, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(dropout_rate))
    model.add(GRU(units // 2, return_sequences=False))
    model.add(Dense(1))

    model.compile(optimizer=optimizer, loss='mean_squared_error')

    early_stop = EarlyStopping(monitor='val_loss', patience=2)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.001)

    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=0, callbacks=[early_stop, reduce_lr])

    test_predict = model.predict(X_test)
    return model, test_predict, history

# Parameters
units = 50
dropout_rate = 0.4
batch_size = 64
epochs = 50
optimizer = 'rmsprop'
look_back = 3

# Dictionary to store MSE for each feature or combination
mse_scores = []

# Step 1: Train GRU model with each feature individually
for feature in selected_features:
    # Prepare data with the current single feature and target feature
    selected_data = data[[target_feature, feature]]
    data_scaled = scaler.fit_transform(selected_data)
    train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]
    
    # Create dataset for GRU
    X_train, y_train = create_dataset(train_data, look_back=look_back)
    X_test, y_test = create_dataset(test_data, look_back=look_back)
    
    # Reshape input data to 3D (samples, timesteps, features) for GRU input
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], X_train.shape[2]))
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], X_test.shape[2]))
    
    # Train GRU model
    model, test_predict, history = train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer)
    
    # Calculate MSE
    mse = mean_squared_error(y_test, test_predict)
    mse_scores.append({"Features": feature, "MSE": mse})

# Step 2: Train GRU model with all 10 features combined
selected_data = data[[target_feature] + selected_features]
data_scaled = scaler.fit_transform(selected_data)
train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]

# Create dataset for GRU with all 10 features
X_train, y_train = create_dataset(train_data, look_back=look_back)
X_test, y_test = create_dataset(test_data, look_back=look_back)

# Reshape input data to 3D (samples, timesteps, features) for GRU input
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], X_train.shape[2]))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], X_test.shape[2]))

# Train GRU model
model, test_predict, history = train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer)

# Calculate MSE
mse = mean_squared_error(y_test, test_predict)
mse_scores.append({"Features": "All Features Combined", "MSE": mse})

# Convert results to DataFrame and sort by MSE
mse_df = pd.DataFrame(mse_scores)
mse_df = mse_df.sort_values(by="MSE").reset_index(drop=True)

# Display the table

print(mse_df)

3/3 [==============================] - 1s 2ms/step
                Features       MSE
0          DL_TRAFFIC_MB  0.002220
1  All Features Combined  0.004987
2      AVG_USR_THRPUT_DL  0.008120


In [21]:
# Load your dataset (adjust the path to where your file is located)
data = pd.read_csv('../../data/Area2.csv')

# List of the first 10 features based on dataset
selected_features = [
    'AVG_NO_USER', 'DL_TRAFFIC_MB'
]

# Target feature
target_feature = 'AVG_USR_THRPUT_DL'

# Normalize all features
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data[[target_feature] + selected_features])

# Split data into train and test sets (80% train, 20% test)
train_size = int(len(data_scaled) * 0.8)
train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]

# Function to create dataset matrix for GRU
def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), :])
        Y.append(dataset[i + look_back, 0])  # Target is the first column after scaling
    return np.array(X), np.array(Y)

# Function to train GRU model
def train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer):
    model = Sequential()
    model.add(GRU(units, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(dropout_rate))
    model.add(GRU(units // 2, return_sequences=False))
    model.add(Dense(1))

    model.compile(optimizer=optimizer, loss='mean_squared_error')

    early_stop = EarlyStopping(monitor='val_loss', patience=2)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.001)

    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=0, callbacks=[early_stop, reduce_lr])

    test_predict = model.predict(X_test)
    return model, test_predict, history

# Parameters
units = 50
dropout_rate = 0.4
batch_size = 64
epochs = 50
optimizer = 'rmsprop'
look_back = 3

# Dictionary to store MSE for each feature or combination
mse_scores = []

# Step 1: Train GRU model with each feature individually
for feature in selected_features:
    # Prepare data with the current single feature and target feature
    selected_data = data[[target_feature, feature]]
    data_scaled = scaler.fit_transform(selected_data)
    train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]
    
    # Create dataset for GRU
    X_train, y_train = create_dataset(train_data, look_back=look_back)
    X_test, y_test = create_dataset(test_data, look_back=look_back)
    
    # Reshape input data to 3D (samples, timesteps, features) for GRU input
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], X_train.shape[2]))
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], X_test.shape[2]))
    
    # Train GRU model
    model, test_predict, history = train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer)
    
    # Calculate MSE
    mse = mean_squared_error(y_test, test_predict)
    mse_scores.append({"Features": feature, "MSE": mse})

# Step 2: Train GRU model with all 10 features combined
selected_data = data[[target_feature] + selected_features]
data_scaled = scaler.fit_transform(selected_data)
train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]

# Create dataset for GRU with all 10 features
X_train, y_train = create_dataset(train_data, look_back=look_back)
X_test, y_test = create_dataset(test_data, look_back=look_back)

# Reshape input data to 3D (samples, timesteps, features) for GRU input
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], X_train.shape[2]))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], X_test.shape[2]))

# Train GRU model
model, test_predict, history = train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer)

# Calculate MSE
mse = mean_squared_error(y_test, test_predict)
mse_scores.append({"Features": "All Features Combined", "MSE": mse})

# Convert results to DataFrame and sort by MSE
mse_df = pd.DataFrame(mse_scores)
mse_df = mse_df.sort_values(by="MSE").reset_index(drop=True)

# Display the table
print(mse_df)

3/3 [==============================] - 1s 3ms/step
                Features       MSE
0  All Features Combined  0.017393
1            AVG_NO_USER  0.017717
2          DL_TRAFFIC_MB  0.024728


In [24]:
# Load your dataset (adjust the path to where your file is located)
data = pd.read_csv('../../data/Area3.csv')

# List of the first 10 features based on dataset
selected_features = [
    'AVG_NO_USER', 'AVG_USR_THRPUT_DL'
]

# Target feature
target_feature = 'DL_TRAFFIC_MB'

# Normalize all features
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data[[target_feature] + selected_features])

# Split data into train and test sets (80% train, 20% test)
train_size = int(len(data_scaled) * 0.8)
train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]

# Function to create dataset matrix for GRU
def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), :])
        Y.append(dataset[i + look_back, 0])  # Target is the first column after scaling
    return np.array(X), np.array(Y)

# Function to train GRU model
def train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer):
    model = Sequential()
    model.add(GRU(units, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(dropout_rate))
    model.add(GRU(units // 2, return_sequences=False))
    model.add(Dense(1))

    model.compile(optimizer=optimizer, loss='mean_squared_error')

    early_stop = EarlyStopping(monitor='val_loss', patience=2)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.001)

    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=0, callbacks=[early_stop, reduce_lr])

    test_predict = model.predict(X_test)
    return model, test_predict, history

# Parameters
units = 50
dropout_rate = 0.4
batch_size = 64
epochs = 50
optimizer = 'rmsprop'
look_back = 3

# Dictionary to store MSE for each feature or combination
mse_scores = []

# Step 1: Train GRU model with each feature individually
for feature in selected_features:
    # Prepare data with the current single feature and target feature
    selected_data = data[[target_feature, feature]]
    data_scaled = scaler.fit_transform(selected_data)
    train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]
    
    # Create dataset for GRU
    X_train, y_train = create_dataset(train_data, look_back=look_back)
    X_test, y_test = create_dataset(test_data, look_back=look_back)
    
    # Reshape input data to 3D (samples, timesteps, features) for GRU input
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], X_train.shape[2]))
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], X_test.shape[2]))
    
    # Train GRU model
    model, test_predict, history = train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer)
    
    # Calculate MSE
    mse = mean_squared_error(y_test, test_predict)
    mse_scores.append({"Features": feature, "MSE": mse})

# Step 2: Train GRU model with all 10 features combined
selected_data = data[[target_feature] + selected_features]
data_scaled = scaler.fit_transform(selected_data)
train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]

# Create dataset for GRU with all 10 features
X_train, y_train = create_dataset(train_data, look_back=look_back)
X_test, y_test = create_dataset(test_data, look_back=look_back)

# Reshape input data to 3D (samples, timesteps, features) for GRU input
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], X_train.shape[2]))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], X_test.shape[2]))

# Train GRU model
model, test_predict, history = train_gru_model(X_train, y_train, X_test, y_test, units, dropout_rate, batch_size, epochs, optimizer)

# Calculate MSE
mse = mean_squared_error(y_test, test_predict)
mse_scores.append({"Features": "All Features Combined", "MSE": mse})

# Convert results to DataFrame and sort by MSE
mse_df = pd.DataFrame(mse_scores)
mse_df = mse_df.sort_values(by="MSE").reset_index(drop=True)

# Display the table
print(mse_df)

3/3 [==============================] - 2s 3ms/step
                Features       MSE
0      AVG_USR_THRPUT_DL  0.005604
1  All Features Combined  0.017738
2            AVG_NO_USER  0.032892
